# Notebook 11: Hard Contrastive Negative Augmentation

## Purpose
Evaluate the effect of adding hard contrastive negatives to the training set.
Hard contrastive negatives are LLM-generated sentences that superficially
resemble hedging language but do not express genuine epistemic uncertainty —
they are designed to sit near the positive manifold in embedding space without
being true hedges.

Unlike positive augmentation (Notebook 10), contrastive negatives:
- Are added to the **negative class** (label=0)
- Do **not** improve class imbalance — the negative class grows slightly
- Target **decision boundary sharpening** — the classifier sees more
  hard examples near the boundary during training

## Generation
1,342 hard contrastive negatives generated by `Llama-3-8B-Instruct`,
prompted on each real positive training example to produce sentences
with hedging surface markers but without genuine epistemic uncertainty.
6 generation failures from 1,348 attempted (674 seeds × 2).

## Filtering Strategy
Identical to Notebook 10 — cosine similarity to nearest real positive
at τ = 0.5, 0.6, 0.7, 0.8. Here the filter selects candidates that are
geometrically close to the positive manifold, which is exactly what we
want for hard negatives — easy negatives far from the boundary provide
no additional signal.

## Hypothesis
Hard negatives near the positive manifold force the classifier to learn
a more precise decision boundary — improving precision at the cost of
potentially reducing recall. The capacity-augmentation interaction
observed in Notebook 10 (MLP-2 benefits more than MLP-1) may not hold
here since we are sharpening rather than expanding the positive class.

## Imbalance
Adding ~270 hard contrastives (τ=0.8 estimate) to 68,836 real negatives
is negligible in terms of imbalance ratio (~102:1 → ~102.5:1). Any
improvement in metrics reflects boundary geometry, not class rebalancing.

## Evaluation
- UMAP: filtered contrastive negatives overlaid on real training data
- Metrics table: precision, recall, F1 vs Notebook 09 and 10 baselines
- DET curves: all conditions overlaid

In [ ]:
# -----------------------------------------------
# Imports and setup — identical to Notebooks 09-10
# -----------------------------------------------
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap.umap_ as umap

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW

from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    det_curve,
)
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# -----------------------------------------------
# Load real training embeddings and splits
# -----------------------------------------------
X_train = np.load("data/processed/embeddings/X_train.npy")
y_train = np.load("data/processed/embeddings/y_train.npy")
X_cal   = np.load("data/processed/embeddings/X_cal.npy")
y_cal   = np.load("data/processed/embeddings/y_cal.npy")
X_test  = np.load("data/processed/embeddings/X_test.npy")
y_test  = np.load("data/processed/embeddings/y_test.npy")

print(f"Train: {X_train.shape} | Positives: {y_train.sum()}")
print(f"Cal:   {X_cal.shape}   | Positives: {y_cal.sum()}")
print(f"Test:  {X_test.shape}  | Positives: {y_test.sum()}")

# -----------------------------------------------
# Load baseline metrics from Notebooks 09 and 10
# -----------------------------------------------
with open("data/results/metrics_09_baseline.json") as f:
    baseline_09 = json.load(f)

with open("data/results/metrics_10_positive_aug.json") as f:
    baseline_10 = json.load(f)

print("\nNotebook 09 baseline:")
for name, m in baseline_09.items():
    print(f"  {name}: F1={m['f1']:.3f} | P={m['precision']:.3f} | R={m['recall']:.3f}")